# PQ ADBC Advisor - Quickstart

Run this notebook inside your Fabric workspace to (1) discover artifacts affected by the ODBC to ADBC connector migration and (2) validate refresh success after you flip the ADBC switch.

This is a **read-only** tool for the scan phase. The validation phase triggers refreshes on the workspace's semantic models.

In [ ]:
%pip install git+https://github.com/MichaelaIsaacs/pq-adbc-advisor.git

## Phase 1 - discovery

Run this **before** you enable ADBC at the tenant or workspace level. It scans every semantic model, dataset, and dataflow in the current workspace for connector calls that fall under the ODBC to ADBC migration, and flags any that explicitly pin `Implementation="1.0"`.

By default, connectors that are **not** part of any migration effort (SQL Server, Excel, Web, etc.) are excluded from the report to keep it focused. Pass `include_non_migrating=True` if you want a full inventory.

In [ ]:
from pq_adbc_advisor import scan_workspace

baseline = scan_workspace()

### Render the report inline

Just type the variable name on its own line - the report renders as a rich, Fabric-branded dashboard right below the cell. No file download needed.

In [ ]:
baseline

### Optional: show the full inventory (including non-migrating connectors)

In [ ]:
baseline.show_non_migrating = True
baseline

### Optional: raw DataFrame view for CSV export

In [ ]:
df = baseline.to_dataframe()
display(df)

## Phase 2 - validation (after the ADBC switch)

Now enable the tenant setting **"Use ADBC drivers for supported connectors"** (or set the workspace override, or remove `Implementation="1.0"` pins).

Then run the cell below. For each impacted semantic model we trigger a fresh refresh, poll for completion, and compare status + duration to the pre-migration baseline. Same inline display pattern.

In [ ]:
from pq_adbc_advisor import validate_migration

result = validate_migration(baseline)
result

## Cost + runtime notes

- The scan is I/O bound; it reads item definitions but does not execute queries.
- On a workspace with ~80 artifacts the scan typically finishes in **under a minute** with the parallel default.
- Adjust `max_parallel` (default 10) up or down depending on your tenant's rate limits.
- No Fabric capacity is consumed by the scan phase. The validation phase consumes capacity because it triggers real refreshes.

## Turn off telemetry (optional)

The first time you scan a workspace you'll see a one-time notice describing what's collected. Disable at any time (choice persists across kernel restarts):

In [ ]:
# from pq_adbc_advisor import disable_telemetry
# disable_telemetry()